In [1]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import FunctionTransformer, OneHotEncoder, OrdinalEncoder, StandardScaler
from sklearn.compose import ColumnTransformer, TransformedTargetRegressor
from sklearn.metrics import root_mean_squared_error
from sklearn.model_selection import GridSearchCV, KFold, train_test_split
from sklearn.pipeline import Pipeline

from xgboost import XGBRegressor
from sklearn.ensemble import RandomForestRegressor
from sklearn.linear_model import ElasticNet, Ridge

In [2]:
df = pd.read_parquet("../data/processed/sparcs_2024_final.parquet")

# Constants Setup

In [3]:
TARGET = "los"
ADMISSION = ["permanent_facility_id","zip_code", "age_group", "gender", "race", 
             "ethnicity","type_of_admission", "emergency_department_indicator","payment_typology_1", "payment_typology_2",
               "payment_typology_3",]

#extra info most likely dropped
POST_ADMISSION = ["ccsr_diagnosis_code", "apr_medical_surgical", "ccsr_procedure_code"]

#features 
ORDINAL_FTRS = ["age_group"]
ORDINAL_CATS = [["0-17", "18-29", "30-49", "50-69", "70 or Older"]]
CATEGORICAL_FTRS = [c for c in ADMISSION + POST_ADMISSION if c not in ORDINAL_FTRS]
CONTINUOUS_FTRS = []


INTERACTION_CANDIDATES = [
    ("type_of_admission", "age_group"),
    ("ccsr_diagnosis_code", "age_group"),
    ("ccsr_diagnosis_code", "type_of_admission"),
    ("type_of_admission", "race"),
]

TRAIN_PCT = 0.10           #Non-leakage features
TRAIN_PCT_POST = 0.05      #W/ leakage features
N_STATES = 5               # Num random states to check consistency
N_FOLDS = 5
TEST_SIZE = 0.20
LOS_BINS = [0, 1, 2, 3, 5, 7, 14, 30, 120]

# Interaction terms

Create a new column for each interaction terms where each value is Column1_cat|Column2_cat. The one hot encoder pre processing creates indicator terms for each combination of categories for interaction terms.

In [4]:
def add_interactions(X, features):
    """Add one combined categorical column per candidate pair whose features are both used"""
    X= X.copy()
    interaction_ftrs = []

    for a, b in INTERACTION_CANDIDATES:
        if a in features and b in features:
            name = f"{a}_x_{b}"
            X[name]= X[a].astype(str)+"|"+ X[b].astype(str)
            interaction_ftrs.append(name)
        
    return X, interaction_ftrs

# Splitting
Splitting is performed for each random state. It splits of a test set and then creates a subset from the train_val set. Returns X_train_val,y_train_val subsets and the X_test y_test

In [5]:
def create_subset(y_train_val, pct, random_state):
    """Stratified (by LOS bin) random subset of train_val, copied from model_training.ipynb"""
    rng = np.random.default_rng(random_state)
    train_bins=pd.cut(y_train_val, bins=LOS_BINS, labels=False).to_numpy()
    selected = []

    for bin_label in np.unique(train_bins):
        idx=rng.permutation(np.where(train_bins == bin_label)[0])
        selected.extend(idx[:max(1, int(round(len(idx) * pct)))])
    
    return rng.permutation(selected)


def split_data(X, y, random_state, train_pct):
    """Stratified train_val/test split, then the CV subset of train_val"""

    X_train_val, X_test, y_train_val, y_test = train_test_split(X, y, test_size=TEST_SIZE,stratify=pd.cut(y, bins=LOS_BINS, labels=False),random_state=random_state)

    subset = create_subset(y_train_val, train_pct, random_state)
    return X_train_val.iloc[subset], y_train_val.iloc[subset], X_test, y_test

# Preprocessing

Ordinal Encoder for the age_group and one hot encoder for the remaining features. Limited to categories with at least 50 samples

In [7]:
def make_preprocessor(ftrs, interaction_ftrs):
    onehot_ftrs = [c for c in ftrs if c in CATEGORICAL_FTRS] + interaction_ftrs
    return ColumnTransformer([
        ("ord", OrdinalEncoder(categories=ORDINAL_CATS), ORDINAL_FTRS),
        ("onehot", OneHotEncoder(min_frequency=50, handle_unknown="infrequent_if_exist"), onehot_ftrs),
    ])

# Models and parameter grids


In [ ]:
#Transformer function for log scaling the target for training model
LOG_Y = FunctionTransformer(np.log1p, inverse_func=np.expm1)

#Two linear models since elastic_net not ideal for ridge
LINEAR_MODELS = ["ridge", "elastic_net"]


def build_model(model_name,ftrs, interaction_ftrs, random_state):
    """Return the pipeline (wrapped so y can be log transformed) and its parameter grid"""
    models = {
        "ridge":Ridge(),
        "elastic_net": ElasticNet(),
        "random_forest": RandomForestRegressor(random_state=random_state),
        "xgboost": XGBRegressor(random_state=random_state),
    }
    pipe = Pipeline([
        ("preprocessor", make_preprocessor(ftrs, interaction_ftrs)),
        ("model", models[model_name]),
    ])
    model = TransformedTargetRegressor(regressor=pipe)

    param_grids = {
        "ridge": {
            "regressor__model__alpha": np.logspace(-1, 5, 7),
            "transformer": [None, LOG_Y],
        },
        "elastic_net": {  
            "regressor__model__alpha": np.logspace(-3, 0, 7),
            "regressor__model__l1_ratio": [0.1, 0.5, 1.0],
            "regressor__model__max_iter": [10000],
            "transformer": [None, LOG_Y],
        },
        "random_forest": {
            "regressor__model__max_depth": [10, 20, 30],
            "regressor__model__max_features": [0.5,0.75, 1.0],
            "regressor__model__n_estimators": [100],
            "regressor__model__n_jobs": [1],
            "transformer": [None],
        },
        "xgboost": {
            "regressor__model__n_estimators": [100, 300, 500],
            "regressor__model__max_depth": [3, 6, 10],
            "regressor__model__colsample_bytree": [0.5, 0.9],
            "regressor__model__learning_rate": [0.1],
            "regressor__model__n_jobs": [1],
            "transformer": [None],
        },
    }
    return model, param_grids[model_name]

# Cross validation

`GridSearchCV` fits the preprocessor on the training folds only, picks the best parameters by mean validation RMSE,
and refits them on the whole CV subset. The test set is scored once with that refit model.

In [9]:
from joblib import parallel_backend


def cross_validation(model, param_grid, X_train_val, y_train_val, X_test, y_test, random_state):
    """Cross Validation function that trains a model choosing its better hyperparameter"""
    kf = KFold(n_splits=N_FOLDS, shuffle=True, random_state=random_state)
    
    grid = GridSearchCV(model, param_grid=param_grid, scoring="neg_root_mean_squared_error",
                        cv=kf, return_train_score=True, n_jobs=-1)
    
    #fix to avoid memory issues with parallel jobs
    with parallel_backend("threading"):
        grid.fit(X_train_val, y_train_val)

    best_model = grid.best_estimator_
    best_val_score = -1* grid.best_score_
    test_score = root_mean_squared_error(y_test, best_model.predict(X_test))
    return best_model, grid.best_params_, best_val_score, test_score,

In [ ]:
def run_experiment(model_name, features, use_interactions=False):
    """Run the whole pipeline for N_STATES random states, print each run and the mean +/- std"""
    X, interaction_ftrs = add_interactions(df[features], features) if use_interactions else (df[features], [])
    y = df[TARGET]

    is_post = bool(set(POST_ADMISSION) & set(features))
    train_pct = TRAIN_PCT_POST if is_post else TRAIN_PCT

    label = f"{model_name} | {'post-admission' if is_post else 'admission'} features | interactions: {interaction_ftrs or 'none'}"
    print(label)

    runs = []
    for random_state in range(10,10*N_STATES+1,10):
        X_train_val, y_train_val, X_test, y_test = split_data(X, y, random_state, train_pct)
        model, param_grid = build_model(model_name, features, interaction_ftrs, random_state)
        best_model,best_params, val_rmse, test_rmse = cross_validation(model, param_grid, X_train_val, y_train_val, X_test, y_test, random_state)
        print(f"random state {random_state:>3} | val RMSE {val_rmse:.4f} | test RMSE {test_rmse:.4f}")
        print(best_params)
        runs.append({"config": label, "random_state": random_state, "val_rmse": val_rmse,
                     "test_rmse": test_rmse, "best_params": str(best_params), "best_model": best_model})

    val, test = np.array([r["val_rmse"] for r in runs]), np.array([r["test_rmse"] for r in runs])
    print(f"val RMSE  {val.mean():.4f} +/- {val.std():.4f}")
    print(f"test RMSE {test.mean():.4f} +/- {test.std():.4f}\n")
    return runs

# Admission features (no leakage)

In [11]:
ridge_res = []
ridge_res.extend(run_experiment("ridge",ADMISSION, use_interactions=False))
ridge_res.extend( run_experiment("ridge", ADMISSION, use_interactions=True))

ridge | admission features | interactions: none
random state   0 | val RMSE 8.5885 | test RMSE 8.5800
{'regressor__model__alpha': np.float64(10.0), 'transformer': None}
random state  42 | val RMSE 8.6555 | test RMSE 8.5063
{'regressor__model__alpha': np.float64(10.0), 'transformer': None}
random state  84 | val RMSE 8.6041 | test RMSE 8.5801
{'regressor__model__alpha': np.float64(10.0), 'transformer': None}
random state 126 | val RMSE 8.6393 | test RMSE 8.5854
{'regressor__model__alpha': np.float64(10.0), 'transformer': None}
random state 168 | val RMSE 8.5968 | test RMSE 8.5419
{'regressor__model__alpha': np.float64(10.0), 'transformer': None}
val RMSE  8.6168 +/- 0.0259
test RMSE 8.5587 +/- 0.0305

ridge | admission features | interactions: ['type_of_admission_x_age_group', 'type_of_admission_x_race']
random state   0 | val RMSE 8.5798 | test RMSE 8.5678
{'regressor__model__alpha': np.float64(10.0), 'transformer': None}
random state  42 | val RMSE 8.6447 | test RMSE 8.4958
{'regresso

In [12]:
elastic_res = []
elastic_res.append(run_experiment("elastic_net",ADMISSION, use_interactions=False))
elastic_res.append(run_experiment("elastic_net", ADMISSION, use_interactions=True))

elastic_net | admission features | interactions: none
random state   0 | val RMSE 8.5885 | test RMSE 8.5799
{'regressor__model__alpha': np.float64(0.0001), 'regressor__model__l1_ratio': 0.5, 'regressor__model__max_iter': 10000, 'transformer': None}
random state  42 | val RMSE 8.6553 | test RMSE 8.5064
{'regressor__model__alpha': np.float64(0.0001), 'regressor__model__l1_ratio': 0.5, 'regressor__model__max_iter': 10000, 'transformer': None}
random state  84 | val RMSE 8.6040 | test RMSE 8.5801
{'regressor__model__alpha': np.float64(0.0001), 'regressor__model__l1_ratio': 0.5, 'regressor__model__max_iter': 10000, 'transformer': None}
random state 126 | val RMSE 8.6392 | test RMSE 8.5852
{'regressor__model__alpha': np.float64(0.0001), 'regressor__model__l1_ratio': 1.0, 'regressor__model__max_iter': 10000, 'transformer': None}
random state 168 | val RMSE 8.5966 | test RMSE 8.5421
{'regressor__model__alpha': np.float64(0.0001), 'regressor__model__l1_ratio': 1.0, 'regressor__model__max_iter':

KeyboardInterrupt: 

In [ ]:
rf_res= run_experiment("random_forest", ADMISSION)

In [ ]:
xgb_res = run_experiment("xgboost", ADMISSION)

# Admission + post-admission features (reported to be leakage)

For comparison only, these features are not known at admission.

In [ ]:
POST_FEATURES = ADMISSION + POST_ADMISSION

post_results = []
post_results += run_experiment("ridge", POST_FEATURES, use_interactions=False)
post_results += run_experiment("ridge", POST_FEATURES, use_interactions=True)
post_results += run_experiment("elastic_net", POST_FEATURES, use_interactions=False)
post_results += run_experiment("elastic_net", POST_FEATURES, use_interactions=True)
post_results += run_experiment("random_forest", POST_FEATURES)
post_results += run_experiment("xgboost", POST_FEATURES)